# 02 — Saldo y posiciones

- Saldo de **toda** la cuenta, y filtrado por un activo.
- En **spot** y en **futuros**, que son bolsas separadas.
- Las **posiciones abiertas**.

Todo lo de este cuaderno va **firmado**: si falla con `10004 Invalid IP`
(Bitunix) o `-2015` (Binance), el problema es la key o la whitelist de IP,
no el código. El cuaderno `00` lo distingue.

### Qué significa exactamente "saldo"

El **disponible**, no el total. Lo bloqueado en órdenes abiertas o retenido
como margen de una posición **no** cuenta. Es la respuesta a "¿con cuánto
puedo abrir ahora mismo?", que es la pregunta que hace una estrategia antes
de entrar.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # el repo, sin instalarlo

from luarix_exchange.brokers.factory import AvailableBrokers, BrokerFactory
from luarix_exchange.brokers.credentials import BrokerCredentials, load_dotenv

load_dotenv()   # lee el .env de la raiz del repo

# Cambia esto para probar el otro broker. Todo lo demas del cuaderno
# funciona igual: esa es justamente la idea del contrato unico.
BROKER   = AvailableBrokers.BITUNIX     # o AvailableBrokers.BINANCE
MARKET   = "futures"                    # o "spot"
SYMBOL   = "BTCUSDT"

# Sin `credentials=`, la factory las lee del .env.
# En produccion el backend las pasa explicitas, una por usuario:
#     BrokerFactory.get_broker(BROKER, credentials=BrokerCredentials(
#         api_key=fila.api_key, api_secret=fila.api_secret))
broker = BrokerFactory.get_broker(
    BROKER,
    testnet=False,          # Bitunix lo ignora: no tiene testnet
    test_order=False,       # solo Binance: True valida sin ejecutar
    trading_mode="live",
    allow_live_trading=False,   # sigue bloqueando el place_order real
)
print(type(broker).__name__, "| futuros:", broker.supports_futures, "| OCO:", broker.supports_oco)

## Toda la cuenta

In [ ]:
saldo = broker.get_balance(market=MARKET)
# Solo lo que tiene algo: una cuenta de Binance devuelve ~400 activos a cero.
{k: v for k, v in sorted(saldo.items(), key=lambda kv: -kv[1]) if v > 0}

## Filtrado por un activo

Un activo pedido y ausente devuelve `{"BTC": 0.0}`, **no** un diccionario
vacío. "No tienes BTC" y "no pregunté por BTC" son respuestas distintas y
quien llama no debería tener que distinguirlas con un `if ... in`.

En los futuros de Bitunix el filtro no recorta un resultado: *es* la
consulta. Su cuenta se pregunta por moneda de margen, de una en una.

In [ ]:
print("USDT:", broker.get_balance(market=MARKET, asset="USDT"))
print("BTC :", broker.get_balance(market=MARKET, asset="BTC"))
print("una moneda que no existe:", broker.get_balance(market=MARKET, asset="NOEXISTE"))

## Spot y futuros son bolsas separadas

In [ ]:
for m in ("spot", "futures"):
    try:
        print(f"{m:8}", broker.get_balance(market=m, asset="USDT"))
    except Exception as e:
        print(f"{m:8} {type(e).__name__}: {e}")

## Posiciones abiertas

`BrokerPosition` tiene la misma forma venga de donde venga, que es todo el
punto: `quantity` **siempre positiva** y la dirección en `side`. Binance
firma la cantidad (`positionAmt` negativo = corto) y Bitunix no; sin
normalizar eso, un `qty > 0` significaría cosas distintas según el exchange.

En **spot** no existen las posiciones: hay saldos de activos. Se devuelven
representados igual (`side="LONG"`, `entry_price=0.0`) para poder listar "lo
que tengo abierto" sin ramificar por mercado. Las monedas de cotización
(USDT, USDC...) se excluyen: tenerlas es tener el efectivo, no estar dentro.

In [ ]:
posiciones = broker.get_positions(market=MARKET)
print(len(posiciones), "posiciones abiertas")
for p in posiciones:
    print(f"  {p.symbol:12} {p.side:5} qty={p.quantity:<12} entrada={p.entry_price:<12} "
          f"marca={p.mark_price:<12} pnl={p.unrealized_pnl:<10} x{p.leverage}")

In [ ]:
# Filtrado por par.
broker.get_positions(market=MARKET, symbol=SYMBOL)

In [ ]:
# El diccionario original del exchange sigue entero, por si hace falta un
# campo que la forma normalizada no expone.
if posiciones:
    posiciones[0].raw

## Apalancamiento (solo futuros)

⚠ Es una operación **de cuenta**, no de orden: afecta a todo lo que se
coloque después en ese par, incluido lo que tenga abierto otra sesión.
Descomenta para ejecutarlo de verdad.

In [ ]:
# broker.set_leverage(SYMBOL, 5)